## **Race Data Cleaning**

##### **Imports**

In [184]:
import pandas as pd
import numpy as np

from pathlib import Path

#### **Retrieving FastF1 Lap Data**

We combine all data files from 2018 to 2025 in a single dataframe for data cleaning and analysis.

In [ ]:
# Import and combine all f1 lap data
laps_directory = Path('../data/ff1_laps_data')
laps_files = laps_directory.glob('laps_data_*.csv')

laps_raw = pd.concat(
    [pd.read_csv(file) for file in laps_files],
    ignore_index=True
)

# laps_raw.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Location,Year
0,0 days 00:08:53.226000,GAS,10,0 days 00:01:45.060000,1.0,NaN,NaN,NaN,NaN,0 days 00:00:25.495000,...,0 days 00:07:07.988000,NaN,NaN,17.0,NaN,NaN,False,False,Melbourne,2018
1,0 days 00:10:26.598000,GAS,10,0 days 00:01:33.372000,2.0,1.0,NaN,NaN,0 days 00:00:31.357000,0 days 00:00:24.825000,...,0 days 00:08:53.226000,NaN,NaN,17.0,NaN,NaN,False,False,Melbourne,2018
2,0 days 00:11:59.459000,GAS,10,0 days 00:01:32.861000,3.0,1.0,NaN,NaN,0 days 00:00:31.160000,0 days 00:00:24.725000,...,0 days 00:10:26.598000,NaN,NaN,17.0,NaN,NaN,False,False,Melbourne,2018
3,0 days 00:13:31.643000,GAS,10,0 days 00:01:32.184000,4.0,1.0,NaN,NaN,0 days 00:00:30.835000,0 days 00:00:24.730000,...,0 days 00:11:59.459000,NaN,NaN,17.0,NaN,NaN,False,False,Melbourne,2018
4,0 days 00:15:03.975000,GAS,10,0 days 00:01:32.332000,5.0,1.0,NaN,NaN,0 days 00:00:30.716000,0 days 00:00:24.821000,...,0 days 00:13:31.643000,NaN,21.0,17.0,NaN,NaN,False,True,Melbourne,2018


### **Standardizing Data Types**

We standardize key data types before analysis to ensure the dataset can be used consistently. 

- Timing columns are converted to timedelta values for accurate time calculations.
- Boolean fields are converted to true boolean types. 
- Entirely blank columns are removed.

In [186]:
# Standardize time data types to timedelta
time_columns = [column for column in laps_raw.keys() if 'Time' in column]
laps_raw[time_columns] = laps_raw[time_columns].apply(pd.to_timedelta, errors='coerce')

In [187]:
# Identify boolean columns
bool_columns = ["IsPersonalBest", "FastF1Generated", "IsAccurate"]
laps_raw[["IsPersonalBest", "FastF1Generated", "IsAccurate"]].dtypes

IsPersonalBest     object
FastF1Generated      bool
IsAccurate           bool
dtype: object

In [188]:
# Update non-boolean values
laps_raw["IsPersonalBest"] = laps_raw["IsPersonalBest"].astype(bool)

In [189]:
# Drop columns with all null values
laps_raw = laps_raw.drop(['Deleted', 'DeletedReason', 'LapStartDate'], axis=1)

### **Handling Missing Data**

A major source of missing data occurs in the `LapTime` column. In the FastF1 dataset, missing lap times appear to be commonly associated with pit laps, retirements, or incomplete laps. However, both 'LapStartTime' and 'Time' have no null values, meaning 'LapTime' can be reconstructed by subtracting the lap start time from the lap end time for every missing row. 

Most of the remaining missing values occur on the first lap for each driver. These missing fields include 'Stint', 'Sector1Time', and 'Sector1SessionTime'. 

- For `Stint`, missing values on lap one are filled with 1. 
- For `Sector1Time`, we subtract 'Sector2Time' and 'Sector3Time' from the full 'LapTime'. 
- For `Sector1SessionTime`, we subtract 'Sector2Time' from 'Sector2SessionTime'.

In [190]:
# Reconstruct missing LapTime
# LapTime = "Time" − "LapStartTime"
mask = laps_raw["LapTime"].isna()
reconstructed_lap_time = laps_raw["Time"] - laps_raw["LapStartTime"]

laps_raw.loc[mask, "LapTime"] = reconstructed_lap_time[mask]

In [191]:
# Find blank column values on the first lap
first_lap = laps_raw["LapNumber"] == 1

# Stint always equals 1 for the first lap
mask = first_lap & laps_raw["Stint"].isna()
laps_raw.loc[mask, "Stint"] = 1

# Sector1Time = LapTime - Sector2Time - Sector3Time
mask = (
    first_lap
    & laps_raw["Sector1Time"].isna()
    & laps_raw["LapTime"].notna()
    & laps_raw["Sector2Time"].notna()
    & laps_raw["Sector3Time"].notna()
)

laps_raw.loc[mask, "Sector1Time"] = (
    laps_raw.loc[mask, "LapTime"]
    - laps_raw.loc[mask, "Sector2Time"]
    - laps_raw.loc[mask, "Sector3Time"]
)

# Sector1SessionTime = Sector2SessionTime - Sector2Time
mask = (
    first_lap
    & laps_raw["Sector1SessionTime"].isna()
    & laps_raw["Sector2SessionTime"].notna()
    & laps_raw["Sector2Time"].notna()
)

laps_raw.loc[mask, "Sector1SessionTime"] = (
    laps_raw.loc[mask, "Sector2SessionTime"]
    - laps_raw.loc[mask, "Sector2Time"]
)

In [192]:
# Assume track status == 1 if no status given
laps_raw.loc[laps_raw['TrackStatus'].isna(), 'TrackStatus'] = 1

### **Standardizing Identities** 

#### **Location Names**

In the FastF1 dataset, certain `Location` values may appear under different names despite referring to the same circuit or race location. To improve consistency across the dataset, we use the replace() method to standardize these names into a single naming convention.

In [193]:
# Normalize location names 
laps_raw['Location'] = laps_raw['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

#### **Team Lineage**

Teams in Formula 1 are frequently renamed over time due to changes in sponsorship, ownership, or manufacturer branding. To maintain consistency across seasons, we will create a dictionary that maps historical team names to their standardized 2025 team names. This normalization process will support more accurate multi-season analysis and allow us to better evaluate team performance over time.

Website for tracking team changes over time: _[Formula 1 Lineage](https://flamingtempura.github.io/formula1-lineage/)_

**Author:** Peter West (FlamingTempura)

In [194]:
team_lineage = {
    # Racing Bulls lineage
    "Toro Rosso": "Racing Bulls",
    "AlphaTauri": "Racing Bulls",
    "RB": "Racing Bulls",
    "Racing Bulls": "Racing Bulls",

    # Kick Sauber lineage
    "Sauber": "Kick Sauber",
    "Alfa Romeo Racing": "Kick Sauber",
    "Alfa Romeo": "Kick Sauber",
    "Kick Sauber": "Kick Sauber",

    # Aston Martin lineage
    "Force India": "Aston Martin",
    "Racing Point": "Aston Martin",
    "Aston Martin": "Aston Martin",

    # Alpine lineage
    "Renault": "Alpine",
    "Alpine": "Alpine",

    # Stable teams
    "McLaren": "McLaren",
    "Williams": "Williams",
    "Haas F1 Team": "Haas",
    "Red Bull Racing": "Red Bull Racing",
    "Mercedes": "Mercedes",
    "Ferrari": "Ferrari",
}

# Standardize names for teams across seasons
laps_raw['Team'] = laps_raw['Team'].apply(lambda x: team_lineage[x])

### **Feature Engineering**

#### **Double Headers**

Some circuits hosted multiple races in the same season, creating duplicate `Location` values with different race start times in `event_location_data.csv`

- Red Bull Ring in Spielberg, Austria hosted a double-header in 2020
- Silverstone Circuit in Silverstone, Great Britain hosted a double-header in 2020
- Bahrain International Circuit in Bahrain, Austria hosted a double-header in 2020
- Red Bull Ring in Spielberg, Austria hosted a double-header in 2021

To handle this, we include `EventName` from the event location file so laps can be grouped more accurately by 'Year', 'Location', 'EventName', and 'Driver'. 

We also merge in `StartTime` to calculate each lap’s exact start time in UTC. This allows us to align lap data with weather data and capture the conditions at the moment each lap occurred.

In [195]:
# Get event location data
event_data = pd.read_csv('../data/event_location_data.csv')

event_data['StartTime'] = pd.to_datetime(event_data['StartTime'])

# Add a race number to each event
event_data = event_data.sort_values(['Year', 'Location', 'StartTime'])
event_data['RaceSeq'] = event_data.groupby(['Year', 'Location']).cumcount()

event_data.loc[event_data['RaceSeq'] == 1]

,EventName,Country,Location,StartTime,Year,lon,lat,RaceSeq
57,Sakhir Grand Prix,Bahrain,Sakhir,2020-12-06 17:10:00,2020,50.512,26.031,1
46,70th Anniversary Grand Prix,Great Britain,Silverstone,2020-08-09 13:10:00,2020,-1.017,52.072,1
43,Styrian Grand Prix,Austria,Spielberg,2020-07-12 13:10:00,2020,14.761,47.223,1
67,Austrian Grand Prix,Austria,Spielberg,2021-07-04 13:00:00,2021,14.761,47.223,1


In [196]:
# Identify when the LapStartTime resets for each driver
new_race = (
    laps_raw.groupby(['Year', 'Location', 'Driver'])['LapStartTime'].diff() < pd.Timedelta(0)
).astype(int)

# Mark races with a sequence number
laps_raw['RaceSeq'] = (
    new_race.groupby([laps_raw['Year'], laps_raw['Location'], laps_raw['Driver']]).cumsum()
)

# Add event name to handle repeat locations
laps_raw = laps_raw.merge(
    event_data[['Year', 'Location', 'RaceSeq', 'EventName', 'StartTime']],
    on=['Year', 'Location', 'RaceSeq'],
    how='left'
)

# Add a UTC lap start time for merging with weather data
laps_raw['LapStartTimeUTC'] = laps_raw['LapStartTime'] + laps_raw['StartTime']

laps_raw = laps_raw.drop(['RaceSeq', 'StartTime'], axis=1)

# laps_raw.sample(5)

Missing `TyreLife` values are reconstructed using neighboring laps within each race and driver grouping

- Function fills missing values forward and backward based on surrounding tire life progression
- Any fully missing groups defaulting to 0.

In [197]:
# Use previous and next rows to fill in tire data
def calculate_tire_life(laps):
    laps = laps.copy()
    
    # Handle trailing NaNs
    for i in range(1, len(laps)):
        if pd.isna(laps.iloc[i]) and pd.notna(laps.iloc[i-1]):
            laps.iloc[i] = laps.iloc[i-1] + 1
            
    laps = laps[::-1] # reverse lap data
    
    # Handle leading NaNs
    for i in range(1, len(laps)):
        if pd.isna(laps.iloc[i]):
            laps.iloc[i] = laps.iloc[i-1] - 1
            
    return laps[::-1] # return original order
        
# Fill in missing tire data
laps_raw['TyreLife'] = (
    laps_raw.groupby(['Year', 'Location', 'EventName', 'Driver'])['TyreLife'].transform(calculate_tire_life)
)

# Does not catch all NaN groups
laps_raw['TyreLife'] = laps_raw['TyreLife'].fillna(0)

We update the `Compound` column to fill missing values.

- Missing values are converted from the string 'nan' to true null values
- Backward fill used to infer the compound from surrounding laps within the same race

In [198]:
# Replace Compound 'nan' string with np.nan
laps_raw['Compound'] = laps_raw['Compound'].replace('nan', np.nan)

# Backward fill missing tire compound values
laps_raw['Compound'] = (
    laps_raw.groupby(['Year', 'Location', 'EventName', 'Driver'])['Compound'].bfill()
)

We create boolean flags to identify pit laps and terminal laps.

- `IsPitLap` marks laps with pit-in or pit-out times
- `IsTerminalLap` identifies each driver’s final recorded lap in a race

In [199]:
# Add pit lap feature
laps_raw["IsPitLap"] = (laps_raw["PitInTime"].notna() | laps_raw["PitOutTime"].notna())

# Add terminal lap feature
laps_raw["IsTerminalLap"] = (
    laps_raw["LapNumber"] == laps_raw.groupby(["Year", "Location", "EventName", "Driver"])["LapNumber"].transform("max")
)

In [ ]:
# Store race data for analysis
laps_raw.to_csv('../data/cleaned_lap_data_2018_2025.csv', index=False)